# 07 — Shipping Mode Performance
Sales, profit, and order volume across all shipping modes.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Shipping Mode Performance ─────────────────────────────────────────────────
ship = fetch_df(cur, """
    SELECT o.ship_mode,
           COUNT(DISTINCT o.order_id)                   AS orders,
           ROUND(SUM(oi.sales),  2)                     AS sales,
           ROUND(SUM(oi.profit), 2)                     AS profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100, 2)  AS margin_pct
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.ship_mode
    ORDER BY sales DESC
""")

bar_w = 0.35
x     = range(len(ship))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

b1 = ax1.bar([i - bar_w/2 for i in x], ship["sales"],
             bar_w, color=PALETTE[0], label="Sales",  edgecolor="white", zorder=3)
b2 = ax1.bar([i + bar_w/2 for i in x], ship["profit"],
             bar_w, color=PALETTE[1], label="Profit", edgecolor="white", zorder=3)
ax2.plot(x, ship["orders"], color=PALETTE[2], marker="o",
         linewidth=2, markersize=7, label="# Orders", zorder=4)

for bar in b1:
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 500,
             fmt_usd(bar.get_height()),
             ha="center", va="bottom", fontsize=8, color=PALETTE[0])
for bar in b2:
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 100,
             fmt_usd(bar.get_height()),
             ha="center", va="bottom", fontsize=8, color=PALETTE[1])
for xi, oi in zip(x, ship["orders"]):
    ax2.text(xi, oi + 10, str(int(oi)),
             ha="center", va="bottom", fontsize=9,
             color=PALETTE[2], fontweight="bold")

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2,
           loc="upper right", frameon=False, fontsize=9)
ax1.set_xticks(list(x))
ax1.set_xticklabels(ship["ship_mode"], fontsize=10)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax1.set_title("Sales by Shipping Mode")
ax1.set_ylabel("USD")
ax2.set_ylabel("# Orders")
ax1.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
